In [1]:
import warnings

In [2]:
warnings.simplefilter(action="ignore")

In [3]:
import pickle
import json
import random

import numpy as np

from lightfm import LightFM
from lightfm.data import Dataset

from replay.metrics import MAP, MRR, RocAuc, HitRate, NDCG, Coverage, Experiment
from replay.splitters import RandomSplitter

In [4]:
random.seed(42)
np.random.seed(42)

In [5]:
USER_COL = "user_name"
ITEM_COL = "item_title"
RATING_COL = "rating"

In [6]:
with open("artificial_profiles_scores.pkl", "rb") as f:
    interactions = pickle.load(f)

with open("artificial_profiles.json", "r", encoding="utf-8") as f:
    features_by_user = json.load(f)

with open("titles_with_tags_dict.pkl", "rb") as f:
    features_by_item = pickle.load(f)

print("Data loaded successfully:")
print(f"  - Users: {len(interactions)}")
print(f"  - Items: {len(features_by_item)}")
print(
    f"  - Total interactions: {sum(len(ratings) for ratings in interactions.values())}"
)

Data loaded successfully:
  - Users: 31
  - Items: 1164
  - Total interactions: 3330


In [7]:
sample_user = list(features_by_user.keys())[0]
print(f"Sample user: {sample_user}")
print(f"Bio: {features_by_user[sample_user]['bio'][:200]}...")
print(f"Tags: {features_by_user[sample_user]['tags']}")

Sample user: global_economics_and_geopolitics_analyst
Bio: A highly analytical researcher deeply engaged in monitoring global economic trends, geopolitical risks, international trade, and the socio-economic development of various regions (especially BRICS, As...
Tags: ['global_economy', 'geopolitics', 'macroeconomics', 'international_economics', 'data_analysis', 'forex', 'regional_development', 'brics', 'political_economy', 'international_relations', 'data_monitoring', 'time_series', 'system_thinking', 'geopolitics_of_BRICS', 'investment_analysis', 'economic_forecasting', 'trade_policy', 'market_forecasting', 'data_mining', 'quantitative_finance', 'global_systems', 'regional_analysis', 'economic_development']


In [8]:
sample_item = list(features_by_item.keys())[0]
print(f"Sample item: {sample_item[:80]}...")
print(f"Tags: {features_by_item[sample_item]}")

Sample item: Исследование приоритетов и механизмов реализации отраслевых (секторальных) полит...
Tags: ['international_relations', 'political_economics', 'policy_analysis', 'BRICS', 'geopolitics', 'international_policy', 'comparative_politics']


In [9]:
import numpy as np
import pandas as pd

In [10]:
data = []

for user in interactions:
    for payload in interactions[user]:
        if interactions[user][payload] > 3:
            data.append(
                (user, json.loads(payload).get("title"), interactions[user][payload])
            )

df = pd.DataFrame(data=data, columns=["user_name", "item_title", "rating"])

In [11]:
df.shape

(2533, 3)

In [12]:
df.head()

,user_name,item_title,rating
0,global_economics_and_geopolitics_analyst,Исследование приоритетов и механизмов реализац...,5
1,global_economics_and_geopolitics_analyst,Сеть военно-политических союзов в Евразии: баз...,4
2,global_economics_and_geopolitics_analyst,Ежемесячный мониторинг мировой экономики и гео...,5
3,global_economics_and_geopolitics_analyst,Евразийская интеграция: правовое измерение,4
4,global_economics_and_geopolitics_analyst,Эволюция политической и деловой элиты в соврем...,4


In [13]:
data = []

for user in features_by_user:
    data.append((user, features_by_user[user]["tags"]))

df_users = pd.DataFrame(data=data, columns=["user_name", "features"])

In [14]:
df_users.head()

,user_name,features
0,global_economics_and_geopolitics_analyst,"[global_economy, geopolitics, macroeconomics, ..."
1,international_relations_and_geopolitics_expert,"[international_politics, geopolitics, regional..."
2,multilingual_linguistics_researcher,"[linguistics, translation_technology, corpus_p..."
3,education_and_cultural_development_expert,"[language_teaching, educational_materials, cul..."
4,sociocultural_anthropologist_and_politician,"[social_anthropology, political_analysis, cult..."


In [15]:
data = []

for item in features_by_item:
    data.append((item, features_by_item[item]))

df_items = pd.DataFrame(data=data, columns=["item_title", "features"])

In [16]:
df_items.head()

,item_title,features
0,Исследование приоритетов и механизмов реализац...,"[international_relations, political_economics,..."
1,Антрополе - научно-популярный видео-подкаст о ...,"[anthropology, social_phenomena, media, podcas..."
2,"Разработка, создание и ведение сайта, посвящен...","[web_development, history, cultural_studies, d..."
3,Перевод с английского языка коллективной моног...,"[criminology, literature_review, translation, ..."
4,Сеть военно-политических союзов в Евразии: баз...,"[geopolitics, international_relations, databas..."


In [17]:
dfg = df.groupby(by=ITEM_COL).agg({USER_COL: "nunique"}).reset_index()
dfg.columns = ["item_title", "user_name_count"]
dfg.sort_values(by="user_name_count", ascending=False)

,item_title,user_name_count
35,«Говори на языке бабушки»: SMM и создание конт...,10
581,Роль БРИКС в переориентации внешней политики Р...,10
710,Трансформация российско-китайского сотрудничес...,9
506,Развитие российско-китайского сотрудничества: ...,9
639,Создание контента для сетевых продюсерских про...,9
...,...,...
276,Кураторы‑наставники: старт в университете,1
624,Современные педагогические технологии для кура...,1
626,Современные тренды государственного управления...,1
627,Содействие команде старшеклассников в разработ...,1


In [18]:
df[USER_COL].nunique()

31

In [19]:
df[ITEM_COL].nunique()

793

In [20]:
train_dfs = []
test_dfs = []

for user_name in sorted(df.user_name.unique()):
    user_train, user_test = RandomSplitter(
        query_column=USER_COL,
        item_column=ITEM_COL,
        test_size=0.2,
    ).split(df[df.user_name == user_name])
    train_dfs.append(user_train)
    test_dfs.append(user_test)

train = pd.concat(train_dfs)
test = pd.concat(test_dfs)

In [21]:
df.shape

(2533, 3)

In [22]:
train.shape

(2027, 3)

In [23]:
train.groupby(by=["user_name"]).agg({"item_title": "nunique"}).reset_index()

,user_name,item_title
0,ai_and_nlp_developer,53
1,ai_policy_and_social_impact_analyst,83
2,cultural_and_media_anthropologist,90
3,cultural_humanities_researcher_media_studies,92
4,data_driven_policy_analyst,51
5,digital_marketing_and_media_strategy,91
6,digital_platform_developer_and_educator,40
7,education_and_cultural_development_expert,86
8,educational_psychologist_and_methodologist,30
9,educational_tech_developer,67


In [24]:
train.user_name.nunique()

31

In [25]:
test.shape

(506, 3)

In [26]:
test.user_name.nunique()

31

In [27]:
test.groupby(by=["user_name"]).agg({"item_title": "nunique"}).reset_index()

,user_name,item_title
0,ai_and_nlp_developer,13
1,ai_policy_and_social_impact_analyst,21
2,cultural_and_media_anthropologist,22
3,cultural_humanities_researcher_media_studies,23
4,data_driven_policy_analyst,13
5,digital_marketing_and_media_strategy,23
6,digital_platform_developer_and_educator,10
7,education_and_cultural_development_expert,22
8,educational_psychologist_and_methodologist,7
9,educational_tech_developer,17


In [28]:
ALL_USERS = train[USER_COL].unique().tolist()
ALL_ITEMS = train[ITEM_COL].unique().tolist()

users = df_users[df_users[USER_COL].isin(ALL_USERS)]
items = df_items[df_items[ITEM_COL].isin(ALL_ITEMS)]

In [29]:
dataset = Dataset()

In [30]:
dataset.fit_partial(ALL_USERS, ALL_ITEMS)

In [31]:
user_mapping = dataset.mapping()[0]
item_mapping = dataset.mapping()[2]

inv_user_mapping = {v: k for k, v in user_mapping.items()}
inv_item_mapping = {v: k for k, v in item_mapping.items()}

In [32]:
train_interactions, train_weights = dataset.build_interactions(
    train[[USER_COL, ITEM_COL, RATING_COL]].values
)

In [33]:
train_interactions

<COOrdinate sparse matrix of dtype 'int32'
	with 2027 stored elements and shape (31, 747)>

In [34]:
train_weights

<COOrdinate sparse matrix of dtype 'float32'
	with 2027 stored elements and shape (31, 747)>

In [35]:
class LightFM4Rec:
    def __init__(
        self,
        model,
        user_mapping,
        item_mapping,
        inv_user_mapping,
        inv_item_mapping,
        user_col,
        item_col,
        rating_col,
    ):
        self.model = model
        self.user_mapping = user_mapping
        self.item_mapping = item_mapping
        self.inv_user_mapping = inv_user_mapping
        self.inv_item_mapping = inv_item_mapping
        self.user_col = user_col
        self.item_col = item_col
        self.rating_col = rating_col

    def fit(
        self,
        rating_matrix,
        train_interactions,
        user_features=None,
        item_features=None,
        epochs=16,
    ):
        self.user_features = user_features
        self.item_features = item_features
        self.train_interactions = train_interactions
        self.model.fit(
            rating_matrix,
            user_features=self.user_features,
            item_features=self.item_features,
            epochs=epochs,
        )

    def _get_lfm_pred(self, user_id):
        pred = self.model.predict(
            user_ids=user_id,
            item_ids=self.item_ids,
            user_features=self.user_features,
            item_features=self.item_features,
        )
        return pred

    def predict(self, test, interaction_matrix, user_col, filter_seen=True, k=10):
        user_ids = test[self.user_col].map(self.user_mapping).unique()
        self.item_ids = list(self.item_mapping.values())

        pred = pd.DataFrame(user_ids, columns=[user_col])
        scores = np.vstack(pred[user_col].apply(self._get_lfm_pred).values)

        if filter_seen:
            scores += np.nan_to_num(interaction_matrix.todense()[user_ids] * -np.inf)

        ind_part = np.argpartition(scores, -k)[:, -k:].copy()
        scores_not_sorted = np.take_along_axis(scores, ind_part, axis=1)
        ind_sorted = np.argsort(scores_not_sorted, axis=1)
        scores_sorted = np.sort(scores_not_sorted, axis=1)
        indices = np.take_along_axis(ind_part, ind_sorted, axis=1)

        preds = pd.DataFrame(
            {
                self.user_col: user_ids,
                self.item_col: np.flip(indices, axis=1).tolist(),
                self.rating_col: np.flip(scores_sorted, axis=1).tolist(),
            }
        ).explode([self.item_col, self.rating_col])
        preds[self.user_col] = preds[self.user_col].map(self.inv_user_mapping)
        preds[self.item_col] = preds[self.item_col].map(self.inv_item_mapping)

        return preds

In [36]:
model = LightFM4Rec(
    LightFM(
        random_state=42,
        loss="warp",
        no_components=16,
    ),
    user_mapping,
    item_mapping,
    inv_user_mapping,
    inv_item_mapping,
    USER_COL,
    ITEM_COL,
    RATING_COL,
)

In [37]:
model.fit(train_weights, train_interactions)

In [38]:
preds_wo_features = model.predict(test, train_interactions, USER_COL)

In [39]:
preds_wo_features[preds_wo_features["user_name"] == "financial_economist_and_analyst"]

,user_name,item_title,rating
10,financial_economist_and_analyst,Оценка интеграционных процессов ЕАЭС (торговый...,0.552123
10,financial_economist_and_analyst,Влияние и эффективность смягчения регулировани...,0.391609
10,financial_economist_and_analyst,Волонтёрство на XIII ежегодной международной к...,0.388788
10,financial_economist_and_analyst,Приближённый поиск ближайших соседей посредств...,0.11266
10,financial_economist_and_analyst,Влияние традиционной и нетрадиционной денежно-...,0.084428
10,financial_economist_and_analyst,Влияние санкционных ограничений на финансовые ...,0.026172
10,financial_economist_and_analyst,Разработка инструмента оценки экономической эф...,0.007544
10,financial_economist_and_analyst,Разработка и развертывание стратегии междунаро...,-0.005913
10,financial_economist_and_analyst,Вывод нового продукта для уже существующего би...,-0.119336
10,financial_economist_and_analyst,Арабский язык для поиска и интерпретации актуа...,-0.149013


In [40]:
user_tags = users["features"].explode().unique()

In [41]:
item_tags = items["features"].explode().unique()

In [42]:
dataset.fit_partial(user_features=user_tags, item_features=item_tags)

In [43]:
sparse_u_features = dataset.build_user_features(
    [[row.user_name, row.features] for row in users.reset_index().itertuples()]
)

In [44]:
sparse_i_features = dataset.build_item_features(
    [[row.item_title, row.features] for row in items.reset_index().itertuples()]
)

In [45]:
model = LightFM4Rec(
    LightFM(
        random_state=42,
        loss="warp",
        no_components=16,
    ),
    user_mapping,
    item_mapping,
    inv_user_mapping,
    inv_item_mapping,
    USER_COL,
    ITEM_COL,
    RATING_COL,
)

In [46]:
model.fit(train_weights, train_interactions, sparse_u_features, sparse_i_features)

In [47]:
preds_w_features = model.predict(test, train_interactions, USER_COL)

In [48]:
preds_w_features[preds_w_features["user_name"] == "financial_economist_and_analyst"]

,user_name,item_title,rating
10,financial_economist_and_analyst,Влияние традиционной и нетрадиционной денежно-...,0.336674
10,financial_economist_and_analyst,Оценка интеграционных процессов ЕАЭС (торговый...,0.021143
10,financial_economist_and_analyst,Влияние санкционных ограничений на финансовые ...,0.019998
10,financial_economist_and_analyst,Арабский язык для поиска и интерпретации актуа...,-0.103566
10,financial_economist_and_analyst,Ежемесячный мониторинг мировой экономики и гео...,-0.116001
10,financial_economist_and_analyst,Влияние и эффективность смягчения регулировани...,-0.165117
10,financial_economist_and_analyst,Мониторинг глобального бизнеса: страновые обзоры,-0.26982
10,financial_economist_and_analyst,Разработка инструмента оценки экономической эф...,-0.274046
10,financial_economist_and_analyst,Анализ влияния экономических и отраслевых факт...,-0.28852
10,financial_economist_and_analyst,Волонтёрство на XIII ежегодной международной к...,-0.367499


In [49]:
K = [10]
metrics = Experiment(
    [
        MAP(K),
        MRR(K),
        RocAuc(10),
        NDCG(K),
        HitRate(K),
        Coverage(K),
    ],
    test,
    train,
    query_column=USER_COL,
    item_column=ITEM_COL,
    rating_column=RATING_COL,
)

In [50]:
metrics.add_result("LFM_wo_features", preds_wo_features)
metrics.add_result("LFM_w_features", preds_w_features)

In [51]:
metrics.results

,MAP@10,MRR@10,RocAuc@10,NDCG@10,HitRate@10,Coverage@10
LFM_wo_features,0.277095,0.603763,0.500339,0.384823,0.838710,0.314592
LFM_w_features,0.411801,0.769892,0.638584,0.531958,0.935484,0.305221


In [52]:
data = []

for user_name in sorted(test.user_name.unique()):
    data.append(
        (
            user_name,
            len(
                set(
                    preds_w_features[
                        preds_w_features["user_name"] == user_name
                    ].item_title.unique()
                ).intersection(test[test["user_name"] == user_name].item_title.unique())
            ),
            test[test["user_name"] == user_name].item_title.nunique(),
        )
    )

check = pd.DataFrame(data=data, columns=["user_name", "intersection", "all_test"])
check["share"] = check.apply(lambda x: x.intersection / x.all_test, axis=1)

In [53]:
check.sort_values(by=["share", "intersection"], ascending=[False, False]).reset_index(
    drop=True
)

,user_name,intersection,all_test,share
0,multilingual_nlp_and_ai_specialist,4,6,0.666667
1,political_science_international_relations_analyst,9,14,0.642857
2,international_relations_and_geopolitics_expert,7,12,0.583333
3,ai_and_nlp_developer,7,13,0.538462
4,geopolitics_and_international_relations_expert,8,16,0.500000
5,regional_studies_and_geopolitics_expert,9,19,0.473684
6,educational_tech_developer,8,17,0.470588
7,multilingual_linguistics_researcher,10,22,0.454545
8,digital_marketing_and_media_strategy,10,23,0.434783
9,social_media_and_marketing_strategist,10,23,0.434783


In [54]:
with open("lightfm_sber_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [55]:
with open("preds_wo_features.pkl", "wb") as f:
    pickle.dump(preds_wo_features, f)

In [56]:
with open("preds_w_features.pkl", "wb") as f:
    pickle.dump(preds_w_features, f)